# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIR⁲ Dataset) Exploration with `mlcroissant`

This notebook provides a step-by-step example for exploring the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library and Python. All dataset items are referenced by their unique Croissant `@id`s.

### Dataset Source
This dataset's Croissant schema is provided at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

The FAIR² dataset captures clinical, comorbidity, treatment, molecular, and anatomical variables for 77 survivors with second primary colorectal cancer.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -q mlcroissant pandas

## 1. Data Loading

Let's load the metadata and records from the dataset using `mlcroissant`. The metadata object allows inspection of the dataset properties and available `recordSet` entries.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset: {metadata.name}\n\nDescription:\n{metadata.description}")

## 2. Data Overview

Let's inspect available **record sets (`cr:RecordSet`)**, fields, and columns, referencing them by their unique Croissant `@id` values for reproducibility. Each record set typically corresponds to a table or main entity in Croissant datasets.

**Note:** For the FAIR² dataset, there is likely a main record set. We'll list its properties as defined in the metadata.

In [ ]:
# List available record sets with their @id, name, and fields
record_sets = list(metadata.record_sets)
if not record_sets:
    # In some croissant schemas, .record_sets may be empty or not rendered properly until records are fetched once
    # Try to trigger schema download and re-assign
    tmp = list(dataset.records())  # This should download and parse the main record set
    record_sets = list(metadata.record_sets)

print("Available Record Sets:")
for rs in record_sets:
    print(f"@id: {rs.id}")
    print(f"  Name: {getattr(rs, 'name', '<no name>')}")
    if hasattr(rs, 'fields'):
        print(f"  Fields:")
        for field in rs.fields:
            print(f"    - @id: {field.id}, Name: {getattr(field, 'name', '<no name>')}, DataType: {getattr(field, 'data_type', '<no type>')}")
    print("")

### Example records preview

Let's print the first two records from the main record set using its Croissant `@id`. Use the `@id` you want from the list above. If only one record set is present, that will be used automatically.

In [ ]:
# Set the main record set @id (update if there are multiple)
main_record_set_id = record_sets[0].id

# Preview records (showing use of @id for reference)
print(f"First 2 records from record set {main_record_set_id}:")
for i, record in enumerate(dataset.records(record_set=main_record_set_id)):
    if i >= 2:
        break
    print(record)


## 3. Data Extraction

We will extract data from the record set(s) into pandas DataFrames for analysis. All record set, field, or column access is by `@id` as listed previously.

In [ ]:
# List of all record set @ids
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for rs_id in record_set_ids:
    # Each record is a dict keyed by field ids
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)

# Example: show columns of the main dataframe referencing by @id
main_df = dataframes[main_record_set_id]
print(f"Columns of main record set ({main_record_set_id}):\n{main_df.columns.tolist()}")
main_df.head()

## 4. Exploratory Data Analysis (EDA)

Let's process the data using field `@id`s. We'll:
- Select a numeric field by its `@id`
- Filter records with a threshold
- Normalize this field
- Optionally group by another field (e.g., sex, if available)

**Tip:** Inspect the data above to determine valid numeric fields and group fields. Their `@id`s were printed above.

In [ ]:
# Inspect columns to pick numeric and grouping fields
main_columns = main_df.columns.tolist()
print("Main columns (by @id):", main_columns)

# Suppose 'cr:Age' is the Kroissant @id for patient age (update if the actual @id differs)
# And suppose 'cr:Sex' is the @id for biological sex (update if different)
numeric_field_id = None
group_field_id = None

# Try to select a likely numeric field from the columns
candidate_numeric = [c for c in main_columns if any(s in c.lower() for s in ['age', 'interval', 'years', 'months', 'duration', 'number', 'count'])]

if candidate_numeric:
    numeric_field_id = candidate_numeric[0]
else:
    # Default to first column for demonstration
    numeric_field_id = main_columns[0]

print(f"Selected numeric field (by @id): {numeric_field_id}")

# Grouping field: sex or comorbidity or anatomical_site for demonstration
candidate_group = [c for c in main_columns if any(x in c.lower() for x in ['sex', 'gender', 'site', 'location', 'comorbidity'])]
if candidate_group:
    group_field_id = candidate_group[0]
else:
    group_field_id = None

if group_field_id:
    print(f"Selected group field (by @id): {group_field_id}")
else:
    print("No obvious group field, grouping will be skipped.")

# Convert the chosen numeric field to numeric, coerce errors
main_df[numeric_field_id] = pd.to_numeric(main_df[numeric_field_id], errors='coerce')

# Filter: e.g., numeric_field > 60 (if age), or > mean
if main_df[numeric_field_id].notnull().any():
    threshold = main_df[numeric_field_id].mean()
    filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
else:
    filtered_df = main_df.copy()

print(f"Filtered records with {numeric_field_id} > {threshold:.1f}:")
display(filtered_df.head())

# Normalize the numeric field
filtered_df[f'{numeric_field_id}_normalized'] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f'{numeric_field_id}_normalized']].head())

# Optionally, group by a categorical field
if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean_'+numeric_field_id)
    print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
    display(grouped_df.head())

## 5. Visualization

Let's visualize the distribution of our chosen numeric field, and, if a group field is available, create grouped bar plots to show means by group. We'll use field and group `@id`s for labeling.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution plot
plt.figure(figsize=(7, 4))
sns.histplot(main_df[numeric_field_id].dropna(), kde=True, bins=8)
plt.title(f"Distribution of '{numeric_field_id}'")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# Grouped plot if grouping is available
if group_field_id and group_field_id in main_df.columns:
    plt.figure(figsize=(8,4))
    sns.barplot(x=group_field_id, y=numeric_field_id, data=main_df, ci=None)
    plt.title(f"Mean of '{numeric_field_id}' by '{group_field_id}'")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion

In this notebook we demonstrated how to load, inspect, and process a tabular clinical dataset using Croissant schemas and the `mlcroissant` Python library:
- All dataset record sets, fields, and columns were referenced by their `@id`.
- We explored field types, filtered and normalized data, and produced basic summary plots.
- These steps are easily reproducible for any dataset published with a Croissant schema.

You can continue your analysis by referencing additional fields and leveraging the full range of `mlcroissant` capabilities.